# Notebook da Tarefa 4 — Ajuste da Árvore a partir do Erro

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Disciplina:** Estrutura de Dados II / Redes de Computadores  
**Equipe:** Grupo 42  
**Integrantes:**  
- Wendell Rodrigues Macedo (Responsável / Scrum Master da Tarefa)  
- Vinicius Firmino Souto  
- Fernando Paiva Coelho  
- João Gustavo Sanches  
- Daniel Dias Paranhos  

---

### Objetivo e Contrato da Tarefa 4
- **Entradas (Intactas):**
  - Ficha de baseline congelada (`data/processed/baseline_por_fluxo.csv`), rótulos e corte temporal causal da Tarefa 2 (`treino_periodo_b.csv`, `validacao_periodo_b.csv`; `teste_periodo_b.csv` permanece fechado).
  - Árvore Inicial da Tarefa 3 (`criterion='gini'`, `max_depth=3`, `min_samples_leaf=50`), sua matriz 3×3 e os `634` erros diagnosticados na validação.
- **O que fazemos aqui:**
  - Mantemos a **mesma família de algoritmo (Árvore de Decisão CART)** e as **mesmas 8 features relativas ao baseline (`COLUNAS_MODELO`)**, sem qualquer coluna de rota, IP ou RTT absoluto.
  - Ajustamos os hiperparâmetros da árvore (`max_depth`, `min_samples_leaf` e `criterion`) na validação, ligando **cada mudança diretamente a um erro concreto da Tarefa 3**.

## 1. Diagnóstico dos Erros da Tarefa 3 e Plano de Ajuste

In [1]:
import csv
import json
from pathlib import Path

DIR_ATUAL = Path(".").resolve()
DIR_PROJETO = DIR_ATUAL.parent if DIR_ATUAL.name.lower() == "notebooks" else DIR_ATUAL
DIR_PROCESSED = DIR_PROJETO / "data" / "processed"
DIR_METADADOS = DIR_PROJETO / "metadados"

with open(DIR_METADADOS / "resumo_tarefa3.json", encoding="utf-8") as f:
    resumo_t3 = json.load(f)

av_t3 = resumo_t3["avaliacao_validacao_oficial"]
erros_t3 = resumo_t3["auditoria_erros_validacao"]["contagem_erros_por_regra"]

print("=== DIAGNÓSTICO DOS 634 ERROS DA TAREFA 3 NA VALIDAÇÃO (32.148 linhas) ===")
print(f"  F1 Macro da Tarefa 3: {av_t3['f1_macro']:.4f} | Recall FALHA: {av_t3['por_classe']['FALHA']['recall']:.4f} | Recall RISCO: {av_t3['por_classe']['RISCO']['recall']:.4f}")
print("  Detalhamento dos erros por regra:")
print(f"    - FALHA->OK (2_n5_timeout>=3): {erros_t3['FALHA->OK (2_n5_timeout>=3)']} erros (sendo 101 em caminhos curtos DE->DE, AR->AR, BR->BR)")
print(f"    - RISCO->OK (5_risco_persistente): {erros_t3['RISCO->OK (5_risco_persistente)']} erros (classe RISCO sumiu inteira na folha OK)")
print(f"    - OK->FALHA (6_ok): {erros_t3['OK->FALHA (6_ok)']} erros (picos isolados com 3.4073 < z_robusto < 3.50 e n5_risco = 1)")

=== DIAGNÓSTICO DOS 634 ERROS DA TAREFA 3 NA VALIDAÇÃO (32.148 linhas) ===
  F1 Macro da Tarefa 3: 0.6546 | Recall FALHA: 0.9617 | Recall RISCO: 0.0000
  Detalhamento dos erros por regra:
    - FALHA->OK (2_n5_timeout>=3): 500 erros (sendo 101 em caminhos curtos DE->DE, AR->AR, BR->BR)
    - RISCO->OK (5_risco_persistente): 131 erros (classe RISCO sumiu inteira na folha OK)
    - OK->FALHA (6_ok): 3 erros (picos isolados com 3.4073 < z_robusto < 3.50 e n5_risco = 1)


## 2. Experimentos de Ajuste na Validação (Tentativas Descartadas vs. Árvore Escolhida)

Todos os ajustes foram treinados (`fit`) **somente no conjunto de treino (`80.386` linhas)** e comparados **somente na validação (`32.148` linhas)**.

In [2]:
with open(DIR_METADADOS / "resumo_tarefa4.json", encoding="utf-8") as f:
    resumo_t4 = json.load(f)

print("=== COMPARAÇÃO DOS EXPERIMENTOS DE AJUSTE NA VALIDAÇÃO ===")
print("Critério | max_depth | min_leaf | Folhas | F1 Macro | Rec FALHA | Rec RISCO | Erros (FAL->OK / RIS->OK / OK->FAL)")
print("-" * 115)
rotulos_curtos = [
    "[Tarefa 3]",
    "[Descartada]",
    "[Descartada]",
    "[Descartada]",
    "[Descartada: 9 folhas]",
    "[ESCOLHIDA: 7 folhas puras]",
]
for exp, tag in zip(resumo_t4["experimentos_validacao"], rotulos_curtos):
    print(
        f"{exp['criterion']:<8} | {exp['max_depth']:9d} | {exp['min_samples_leaf']:8d} | "
        f"{exp['n_folhas']:6d} | {exp['f1_macro']:8.4f} | {exp['recall_falha']:9.4f} | "
        f"{exp['recall_risco']:9.4f} | {exp['total_erros']:5d} ({exp['erros_falha_em_ok']:3d} / {exp['erros_risco_em_ok']:3d} / {exp['erros_ok_em_falha']:1d}) {tag}"
    )

exp_f9 = resumo_t4["experimento_feature_nova"]
print("\nExperimento com 9ª coluna relativa candidata (delta_z_robusto):")
print(f"  F1 Macro = {exp_f9['f1_macro_val']:.4f} com {exp_f9['n_folhas']} folhas -> {exp_f9['decisao']}")

=== COMPARAÇÃO DOS EXPERIMENTOS DE AJUSTE NA VALIDAÇÃO ===
Critério | max_depth | min_leaf | Folhas | F1 Macro | Rec FALHA | Rec RISCO | Erros (FAL->OK / RIS->OK / OK->FAL)
-------------------------------------------------------------------------------------------------------------------
gini     |         3 |       50 |      5 |   0.6546 |    0.9617 |    0.0000 |   634 (500 / 131 / 3) [Tarefa 3]
gini     |         4 |       50 |      6 |   0.6655 |    1.0000 |    0.0000 |   134 (  0 / 131 / 3) [Descartada]
gini     |         5 |       20 |      7 |   0.6655 |    1.0000 |    0.0000 |   134 (  0 / 131 / 3) [Descartada]
gini     |         6 |       50 |      8 |   0.9999 |    1.0000 |    1.0000 |     3 (  0 /   0 / 3) [Descartada]
gini     |         6 |        5 |      9 |   1.0000 |    1.0000 |    1.0000 |     0 (  0 /   0 / 0) [Descartada: 9 folhas]
entropy  |         6 |       20 |      7 |   1.0000 |    1.0000 |    1.0000 |     0 (  0 /   0 / 0) [ESCOLHIDA: 7 folhas puras]

Experimen

## 3. Estrutura da Árvore Ajustada Oficial e Novas Regras em Português

In [3]:
arv_aj = resumo_t4["arvore_ajustada"]
print("=== ÁRVORE DE DECISÃO AJUSTADA (TAREFA 4) ===")
print("Critério: entropy | max_depth pedido: 6 | min_samples_leaf: 20 | Semente: 42")
print(f"Profundidade real obtida: {arv_aj['profundidade_real_obtida']} | Número de folhas: {arv_aj['numero_folhas']} | Total de nós: {arv_aj['numero_nos_totais']}\n")
print(arv_aj["diagrama_texto"])

print("\n=== AS 7 REGRAS DA ÁRVORE AJUSTADA (EM PORTUGUÊS) ===")
for idx, r in enumerate(arv_aj["regras_portugues"], 1):
    print(f"Regra {idx}: {r['regra_pt']} (n_treino={r['n_treino']})")

=== ÁRVORE DE DECISÃO AJUSTADA (TAREFA 4) ===
Critério: entropy | max_depth pedido: 6 | min_samples_leaf: 20 | Semente: 42
Profundidade real obtida: 6 | Número de folhas: 7 | Total de nós: 13

|--- z_robusto <= 3.5010 (n=80386, entropy=1.0058)
|   |--- perda_pct <= 16.6650 (n=60817, entropy=0.8028)
|   |   |--- n5_aumento80 <= 1.5000 (n=50520, entropy=0.3902)
|   |   |   |--- n5_timeout <= 2.5000 (n=48629, entropy=0.2319)
|   |   |   |   |--- z_robusto <= 2.0003 (n=47257, entropy=0.0479)
|   |   |   |   |   |--- classe: OK (n=46212, [OK=46212, RISCO=0, FALHA=0], entropy=-0.0000)
|   |   |   |   |--- z_robusto >  2.0003
|   |   |   |   |   |--- n5_risco <= 1.5000 (n=1045, entropy=0.7969)
|   |   |   |   |   |   |--- classe: OK (n=793, [OK=793, RISCO=0, FALHA=0], entropy=-0.0000)
|   |   |   |   |   |--- n5_risco >  1.5000
|   |   |   |   |   |   |--- classe: RISCO (n=252, [OK=0, RISCO=252, FALHA=0], entropy=-0.0000)
|   |   |   |--- n5_timeout >  2.5000
|   |   |   |   |--- classe: FALH

## 4. Comparação Tarefa 3 → Tarefa 4 e Matriz 3×3 na Validação

In [4]:
comp = resumo_t4["comparacao_tarefa3_tarefa4"]
t3, t4, g = comp["tarefa3"], comp["tarefa4"], comp["ganho_absoluto"]

print("=== TABELA COMPARATIVA NA VALIDAÇÃO (TAREFA 3 -> TAREFA 4) ===")
print("Modelo                 | F1 Macro | Recall FALHA | Recall RISCO | Folhas | Erros na Validação")
print("-" * 93)
print(f"Árvore da Tarefa 3     |   {t3['f1_macro']:.4f} |       {t3['recall_falha']:.4f} |       {t3['recall_risco']:.4f} | {t3['n_folhas']:6d} | {t3['total_erros_val']:18d}")
print(f"Árvore desta Tarefa 4  |   {t4['f1_macro']:.4f} |       {t4['recall_falha']:.4f} |       {t4['recall_risco']:.4f} | {t4['n_folhas']:6d} | {t4['total_erros_val']:18d}")
print(f"Ganho Absoluto         |  +{g['delta_f1_macro']:.4f} |      +{g['delta_recall_falha']:.4f} |      +{g['delta_recall_risco']:.4f} |     +2 |              -{g['erros_eliminados_val']:4d}\n")

cm4 = resumo_t4["avaliacao_validacao_ajustada"]["matriz_confusao"]
print("=== MATRIZ DE CONFUSÃO 3x3 DA ÁRVORE AJUSTADA NA VALIDAÇÃO (32.148 medições) ===")
print("Verdadeiro \\ Previsto |       OK |    RISCO |    FALHA |    Total")
print("-" * 65)
for c_t in ["OK", "RISCO", "FALHA"]:
    tot = sum(cm4[c_t].values())
    print(f"{c_t:<21} | {cm4[c_t]['OK']:>8,d} | {cm4[c_t]['RISCO']:>8,d} | {cm4[c_t]['FALHA']:>8,d} | {tot:>8,d}".replace(",", "."))

print("\n=== VERIFICAÇÃO DOS 3 CASOS QUE A TAREFA 3 HAVIA ERRADO ===")
for ex in resumo_t4["auditoria_validacao_ajustada"]["exemplos_corrigidos"]:
    print(f"  - {ex['caso']}: fluxo={ex['fluxo_id']} | Real={ex['classe_verdadeira']} | T3={ex['classe_tarefa3']} -> T4={ex['classe_tarefa4']}")

=== TABELA COMPARATIVA NA VALIDAÇÃO (TAREFA 3 -> TAREFA 4) ===
Modelo                 | F1 Macro | Recall FALHA | Recall RISCO | Folhas | Erros na Validação
---------------------------------------------------------------------------------------------
Árvore da Tarefa 3     |   0.6546 |       0.9617 |       0.0000 |      5 |                634
Árvore desta Tarefa 4  |   1.0000 |       1.0000 |       1.0000 |      7 |                  0
Ganho Absoluto         |  +0.3454 |      +0.0383 |      +1.0000 |     +2 |               -634

=== MATRIZ DE CONFUSÃO 3x3 DA ÁRVORE AJUSTADA NA VALIDAÇÃO (32.148 medições) ===
Verdadeiro \ Previsto |       OK |    RISCO |    FALHA |    Total
-----------------------------------------------------------------
OK                    |   18.975 |        0 |        0 |   18.975
RISCO                 |        0 |      131 |        0 |      131
FALHA                 |        0 |        0 |   13.042 |   13.042

=== VERIFICAÇÃO DOS 3 CASOS QUE A TAREFA 3 HAVIA ERRAD